# 02 — BigQuery Org Graph

Replaces the deprecated AlloyDB + Apache AGE notebook with **BigQuery** as the
relational store for the OrgIQ org graph.

| Concern | Solution |
|---|---|
| Tables | `people`, `teams`, `skills`, `team_memberships`, `person_skills`, `collaborations`, `projects`, `project_contributions` |
| Queries | Standard SQL via `google-cloud-bigquery` |
| Idle cost | $0 — storage-only when idle (no compute floor) |
| Access | Application Default Credentials (notebook 01) |

**AWS counterpart:** `notebooks/aws/02-aurora-graph.ipynb` (Aurora Serverless v2
+ Apache AGE — Cypher via RDS Data API). On GCP we model the same graph as
relational BigQuery tables; downstream notebooks join against these.

Seed data lives under `data/org/` (`people.json`, `teams.json`,
`skills-taxonomy.json`).

## Step 0 — Imports + config

Reads `gcp.project_id`, `gcp.bq_dataset` (default `orgiq`), and
`gcp.bq_location` (default `US`) from `config.json`. Initialises a BigQuery
client and the shared `ResourceTracker` so 19-cleanup can drain the dataset.

In [1]:
import json
from datetime import date, datetime, timezone
from pathlib import Path

from google.cloud import bigquery

from cleanup_utils import ResourceTracker

CONFIG_PATH = Path("config.json")
config = json.loads(CONFIG_PATH.read_text()) if CONFIG_PATH.exists() else {}
gcp_cfg = config.setdefault("gcp", {})

PROJECT_ID  = gcp_cfg.get("project_id")
BQ_DATASET  = gcp_cfg.get("bq_dataset", "orgiq")
BQ_LOCATION = gcp_cfg.get("bq_location", "US")

if not PROJECT_ID:
    raise SystemExit(
        "✗ gcp.project_id missing in config.json. Run notebook 01 first to set\n"
        "  Application Default Credentials and seed the GCP config block."
    )

# Persist the dataset/location choice so other notebooks pick it up.
gcp_cfg["bq_dataset"] = BQ_DATASET
gcp_cfg["bq_location"] = BQ_LOCATION
CONFIG_PATH.write_text(json.dumps(config, indent=2))

bq = bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)
tracker = ResourceTracker()

DATASET_REF = f"`{PROJECT_ID}.{BQ_DATASET}`"

print(f"✓ BigQuery client ready")
print(f"  Project : {PROJECT_ID}")
print(f"  Dataset : {BQ_DATASET}")
print(f"  Location: {BQ_LOCATION}")

✓ BigQuery client ready
  Project : hoehncloudconsole
  Dataset : orgiq
  Location: US


## Step 1 — Create the BigQuery dataset

`create_dataset(..., exists_ok=True)` is idempotent. We tag it in the
`ResourceTracker` so 19-cleanup can drop it via `bq rm -r -f -d`.

In [2]:
dataset = bigquery.Dataset(f"{PROJECT_ID}.{BQ_DATASET}")
dataset.location = BQ_LOCATION
dataset.description = "OrgIQ org graph (people, teams, skills, edges) — GCP track"

dataset = bq.create_dataset(dataset, exists_ok=True)
print(f"✓ Dataset ready: {dataset.full_dataset_id} ({dataset.location})")

tracker.track(
    "bigquery_dataset",
    BQ_DATASET,
    project=PROJECT_ID,
    location=BQ_LOCATION,
)
tracker.flush()
print(f"  Tracked for cleanup: bigquery_dataset/{BQ_DATASET}")

✓ Dataset ready: hoehncloudconsole:orgiq (US)
  Tracked for cleanup: bigquery_dataset/orgiq


## Step 2 — Create the 8 graph tables

Adapts the Postgres DDL in `schema/orgiq_schema.sql` to BigQuery types
(`STRING`, `FLOAT64`, `TIMESTAMP`, `DATE`). `CREATE TABLE IF NOT EXISTS` keeps
the cell idempotent. Each table is clustered on its natural key for fast
lookup at workshop scale.

In [3]:
DDL_STATEMENTS = [
    f"""
    CREATE TABLE IF NOT EXISTS {DATASET_REF}.people (
      id STRING NOT NULL,
      name STRING NOT NULL,
      email STRING,
      title STRING,
      department STRING,
      hire_date DATE,
      manager_id STRING,
      expertise_score FLOAT64,
      pagerank_score FLOAT64,
      composite_score FLOAT64,
      centrality_score FLOAT64,
      betweenness_score FLOAT64,
      scored_at TIMESTAMP,
      updated_at TIMESTAMP
    ) CLUSTER BY id
    """,
    f"""
    CREATE TABLE IF NOT EXISTS {DATASET_REF}.teams (
      id STRING NOT NULL,
      name STRING NOT NULL,
      org_path STRING,
      charter STRING,
      parent_id STRING
    ) CLUSTER BY id
    """,
    f"""
    CREATE TABLE IF NOT EXISTS {DATASET_REF}.skills (
      id STRING NOT NULL,
      name STRING NOT NULL,
      category STRING,
      subcategory STRING
    ) CLUSTER BY id
    """,
    f"""
    CREATE TABLE IF NOT EXISTS {DATASET_REF}.team_memberships (
      person_id STRING,
      team_id STRING,
      role STRING,
      weight FLOAT64
    ) CLUSTER BY person_id, team_id
    """,
    f"""
    CREATE TABLE IF NOT EXISTS {DATASET_REF}.person_skills (
      person_id STRING,
      skill_id STRING,
      weight FLOAT64,
      source STRING
    ) CLUSTER BY person_id, skill_id
    """,
    f"""
    CREATE TABLE IF NOT EXISTS {DATASET_REF}.collaborations (
      person_a_id STRING,
      person_b_id STRING,
      weight FLOAT64,
      last_seen TIMESTAMP
    ) CLUSTER BY person_a_id, person_b_id
    """,
    f"""
    CREATE TABLE IF NOT EXISTS {DATASET_REF}.projects (
      id STRING,
      name STRING,
      status STRING
    )
    """,
    f"""
    CREATE TABLE IF NOT EXISTS {DATASET_REF}.project_contributions (
      person_id STRING,
      project_id STRING,
      role STRING,
      weight FLOAT64
    )
    """,
]

for ddl in DDL_STATEMENTS:
    bq.query(ddl).result()

tables = sorted(t.table_id for t in bq.list_tables(f"{PROJECT_ID}.{BQ_DATASET}"))
print(f"✓ {len(tables)} tables in {BQ_DATASET}:")
for name in tables:
    print(f"  - {name}")

✓ 8 tables in orgiq:
  - collaborations
  - people
  - person_skills
  - project_contributions
  - projects
  - skills
  - team_memberships
  - teams


## Step 3 — Load `people`

Reads `data/org/people.json` (20 records), normalises the `hire_date` to a
`date` object, stamps `updated_at`, and `WRITE_TRUNCATE`s the `people` table
so reruns are idempotent. Score columns stay null until notebook 14.

In [4]:
people_path = Path("data/org/people.json")
people_raw = json.loads(people_path.read_text())
now_iso = datetime.now(timezone.utc).isoformat()

people_rows = [
    {
        "id": p["id"],
        "name": p["name"],
        "email": p.get("email"),
        "title": p.get("title"),
        "department": p.get("department"),
        "hire_date": p.get("hire_date"),
        "manager_id": p.get("manager_id"),
        "updated_at": now_iso,
    }
    for p in people_raw
]

people_schema = [
    bigquery.SchemaField("id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("name", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("email", "STRING"),
    bigquery.SchemaField("title", "STRING"),
    bigquery.SchemaField("department", "STRING"),
    bigquery.SchemaField("hire_date", "DATE"),
    bigquery.SchemaField("manager_id", "STRING"),
    bigquery.SchemaField("updated_at", "TIMESTAMP"),
]

job = bq.load_table_from_json(
    people_rows,
    f"{PROJECT_ID}.{BQ_DATASET}.people",
    job_config=bigquery.LoadJobConfig(
        schema=people_schema,
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    ),
)
job.result()
print(f"✓ Loaded {len(people_rows)} people into {BQ_DATASET}.people")

✓ Loaded 20 people into orgiq.people


## Step 4 — Load `teams` and derive `team_memberships`

`teams.json` carries the org chart plus a `member_ids[]` array per team. We
load the team rows directly, then expand `member_ids` into edges in
`team_memberships` with `role='member'` and `weight=1.0`.

In [5]:
teams_path = Path("data/org/teams.json")
teams_raw = json.loads(teams_path.read_text())

team_rows = [
    {
        "id": t["id"],
        "name": t["name"],
        "org_path": t.get("org_path"),
        "charter": t.get("charter"),
        "parent_id": t.get("parent_id"),
    }
    for t in teams_raw
]

team_schema = [
    bigquery.SchemaField("id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("name", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("org_path", "STRING"),
    bigquery.SchemaField("charter", "STRING"),
    bigquery.SchemaField("parent_id", "STRING"),
]

bq.load_table_from_json(
    team_rows,
    f"{PROJECT_ID}.{BQ_DATASET}.teams",
    job_config=bigquery.LoadJobConfig(
        schema=team_schema,
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    ),
).result()
print(f"✓ Loaded {len(team_rows)} teams into {BQ_DATASET}.teams")

# Derive team_memberships from each team's member_ids[].
membership_rows = []
for t in teams_raw:
    for pid in t.get("member_ids", []) or []:
        membership_rows.append({
            "person_id": pid,
            "team_id": t["id"],
            "role": "member",
            "weight": 1.0,
        })

membership_schema = [
    bigquery.SchemaField("person_id", "STRING"),
    bigquery.SchemaField("team_id", "STRING"),
    bigquery.SchemaField("role", "STRING"),
    bigquery.SchemaField("weight", "FLOAT64"),
]

bq.load_table_from_json(
    membership_rows,
    f"{PROJECT_ID}.{BQ_DATASET}.team_memberships",
    job_config=bigquery.LoadJobConfig(
        schema=membership_schema,
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    ),
).result()
print(f"✓ Loaded {len(membership_rows)} edges into {BQ_DATASET}.team_memberships")

✓ Loaded 4 teams into orgiq.teams
✓ Loaded 20 edges into orgiq.team_memberships


## Step 5 — Load `skills` and synthesise light `person_skills` edges

Flattens `skills-taxonomy.json` (categories → skills) into the `skills` table.

**Synthesis note.** The seed data does not include person→skill edges, so we
generate a small deterministic mock by mapping each person's `department` to
a primary skill category (Platform → DevOps + Backend, Product → Product +
Frontend, Data Platform → Data + Backend, Design Systems → Frontend +
Product). Every person gets 5–7 edges with weights tapering from 0.95 → 0.55.
Real person→skill scoring comes from `notebook 14-scoring` once the synthetic
corpus is generated; this seed is just enough to keep downstream joins
non-empty.

In [6]:
skills_path = Path("data/org/skills-taxonomy.json")
skills_raw = json.loads(skills_path.read_text())

# Flatten the taxonomy into skills rows.
skill_rows = []
for category in skills_raw.get("categories", []):
    cat_name = category["name"]
    for skill in category.get("skills", []):
        skill_rows.append({
            "id": skill["id"],
            "name": skill["name"],
            "category": cat_name,
            "subcategory": skill.get("subcategory"),
        })

skill_schema = [
    bigquery.SchemaField("id", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("name", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("category", "STRING"),
    bigquery.SchemaField("subcategory", "STRING"),
]

bq.load_table_from_json(
    skill_rows,
    f"{PROJECT_ID}.{BQ_DATASET}.skills",
    job_config=bigquery.LoadJobConfig(
        schema=skill_schema,
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    ),
).result()
print(f"✓ Loaded {len(skill_rows)} skills into {BQ_DATASET}.skills")

# Department → ordered list of skill categories (deterministic synth).
DEPT_TO_CATEGORIES = {
    "Platform Engineering": ["DevOps", "Backend"],
    "Product":               ["Product", "Frontend"],
    "Data Platform":         ["Data", "Backend"],
    "Design Systems":        ["Frontend", "Product"],
}

# Index skills by category for the mock.
skills_by_category: dict[str, list[dict]] = {}
for s in skill_rows:
    skills_by_category.setdefault(s["category"], []).append(s)

WEIGHT_TAPER = [0.95, 0.88, 0.80, 0.72, 0.65, 0.60, 0.55]

person_skill_rows = []
for person in people_raw:
    dept = person.get("department") or ""
    categories = DEPT_TO_CATEGORIES.get(dept, ["Backend"])
    # Pick skills round-robin from the dept's categories until we hit 6.
    picked: list[dict] = []
    cursors = {c: 0 for c in categories}
    while len(picked) < 6:
        progressed = False
        for c in categories:
            pool = skills_by_category.get(c, [])
            idx = cursors[c]
            if idx < len(pool):
                picked.append(pool[idx])
                cursors[c] = idx + 1
                progressed = True
                if len(picked) >= 6:
                    break
        if not progressed:
            break
    for rank, skill in enumerate(picked):
        person_skill_rows.append({
            "person_id": person["id"],
            "skill_id": skill["id"],
            "weight": WEIGHT_TAPER[min(rank, len(WEIGHT_TAPER) - 1)],
            "source": "seed-mock",
        })

person_skill_schema = [
    bigquery.SchemaField("person_id", "STRING"),
    bigquery.SchemaField("skill_id", "STRING"),
    bigquery.SchemaField("weight", "FLOAT64"),
    bigquery.SchemaField("source", "STRING"),
]

bq.load_table_from_json(
    person_skill_rows,
    f"{PROJECT_ID}.{BQ_DATASET}.person_skills",
    job_config=bigquery.LoadJobConfig(
        schema=person_skill_schema,
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    ),
).result()
print(f"✓ Loaded {len(person_skill_rows)} edges into {BQ_DATASET}.person_skills")
print("  (synthetic, source='seed-mock' — replaced by notebook 14)")

✓ Loaded 29 skills into orgiq.skills
✓ Loaded 120 edges into orgiq.person_skills
  (synthetic, source='seed-mock' — replaced by notebook 14)


### Projects — left empty

`data/org/projects.json` does not ship with the workshop seed, so the
`projects` and `project_contributions` tables stay empty here. They are
populated later by the synthetic-corpus pipeline (notebook 03 → RAG corpus
ingestion) or by hand for live demos.

## Step 6 — Sanity-check queries

Three small `SELECT`s that verify the graph: total people, the top-3
most-skilled people, and a small sample of skill overlaps across departments.

In [7]:
# 1. People count.
people_count_sql = f"SELECT COUNT(*) AS n FROM {DATASET_REF}.people"
people_count = next(iter(bq.query(people_count_sql).result())).n
print(f"people count: {people_count}")

# 2. Top-3 most-skilled people (by # of person_skills edges).
top_skilled_sql = f"""
SELECT
  ps.person_id,
  ANY_VALUE(p.name)       AS name,
  ANY_VALUE(p.department) AS department,
  COUNT(*)                AS skill_edges
FROM {DATASET_REF}.person_skills ps
JOIN {DATASET_REF}.people p ON p.id = ps.person_id
GROUP BY ps.person_id
ORDER BY skill_edges DESC, ps.person_id
LIMIT 3
"""
print("\nTop-3 most-skilled people:")
for row in bq.query(top_skilled_sql).result():
    print(f"  {row.person_id:6s} {row.name:20s} {row.department:22s} edges={row.skill_edges}")

# 3. Skills shared across departments (sample 5).
cross_dept_sql = f"""
SELECT
  s.name                            AS skill,
  COUNT(DISTINCT p.department)      AS dept_count,
  ARRAY_AGG(DISTINCT p.department
            ORDER BY p.department
            LIMIT 5)                AS departments
FROM {DATASET_REF}.person_skills ps
JOIN {DATASET_REF}.people p ON p.id = ps.person_id
JOIN {DATASET_REF}.skills s ON s.id = ps.skill_id
GROUP BY s.name
HAVING dept_count > 1
ORDER BY dept_count DESC, skill
LIMIT 5
"""
print("\nSkills shared across departments (sample 5):")
for row in bq.query(cross_dept_sql).result():
    print(f"  {row.skill:30s} depts={row.dept_count} -> {list(row.departments)}")

people count: 20

Top-3 most-skilled people:
  p-001  Alex Chen            Platform Engineering   edges=6
  p-002  Maria Santos         Platform Engineering   edges=6
  p-003  James Wright         Platform Engineering   edges=6

Skills shared across departments (sample 5):
  API Design                     depts=2 -> ['Data Platform', 'Platform Engineering']
  Accessibility                  depts=2 -> ['Design Systems', 'Product']
  CSS/Tailwind                   depts=2 -> ['Design Systems', 'Product']
  Distributed Systems            depts=2 -> ['Data Platform', 'Platform Engineering']
  Python                         depts=2 -> ['Data Platform', 'Platform Engineering']


## Summary

| Resource | Details |
|---|---|
| **Store** | BigQuery dataset `{project}.{dataset}` |
| **Tables** | `people`, `teams`, `skills`, `team_memberships`, `person_skills`, `collaborations`, `projects`, `project_contributions` |
| **Seed source** | `data/org/people.json`, `data/org/teams.json`, `data/org/skills-taxonomy.json` |
| **Person→skill edges** | Synthetic, `source='seed-mock'`; rescored by notebook 14 |
| **Idle cost** | $0 (storage-only when idle) |
| **Cleanup** | Dataset tracked under `gcp.tracked_resources` → 19-cleanup `bq rm -r -f -d` |

**Next:** Notebook **03** ingests the org corpus into a Vertex AI RAG Engine
corpus; notebook **14** writes real expertise / PageRank / composite scores
into the `people` table.